# 🧠 ZUCE Colab: Qwen3 / Qwen2.5 Capability-Aware Model Slicing & Quantization (INT8 / INT4)

Notebook นี้ใช้ **ZUCE (Zero-Update Capability Extraction)** เพื่อสกัดเฉพาะความสามารถที่ต้องการ (เช่น Coding หรือ Math) จากโมเดลตระกูล **Qwen3 / Qwen2.5** พร้อม **Quantization 8-bit & 4-bit (NF4)** โดย **ไม่ Fine-tune และไม่แตะค่าน้ำหนักเดิมแม้แต่บิตเดียว ($\Delta \theta = 0$)**

💡 **Double Compression (ZUCE + 4-bit Quant)**: ปรับลดขนาดด้วย % Reduction Slider และทำ 4-bit Quantization เพื่อให้รันโมเดลขนาดใหญ่บน GPU เล็กได้อย่างลื่นไหล

In [ ]:
!nvidia-smi
!pip -q install -U "git+https://github.com/YangNobody12/ZUCE.git" "transformers>=4.51.0" accelerate bitsandbytes safetensors sentencepiece "protobuf>=5.29.1,<6" huggingface_hub
# If Colab asks to restart the runtime after dependency changes, restart once and continue from this notebook.

In [ ]:
# Optional: login if you use a gated/private model or want a more stable HF download session.
from huggingface_hub import notebook_login

notebook_login()

## ⚙️ กำหนดโมเดลและสัดส่วนที่ต้องการลด (% Reduction)
เลือกรุ่นโมเดลที่ต้องการ และปรับแถบเลื่อน (Slider) เปอร์เซ็นต์ที่ต้องการลดขนาด:
- **15% (Noise Pruning)**: ตัดนิวรอนรบกวนออก ประสิทธิภาพดีขึ้น 100-120%
- **35% (Optimal Specialist - แนะนำ)**: ขนาดลดลง ~35% คงความฉลาดในโดเมนเป้าหมายได้สมบูรณ์แบบ
- **45% (Aggressive Boundary)**: ขอบเขตล่างสุดก่อนเกิด Representation Drift

In [ ]:
#@title 🎛️ Extraction Settings (ปรับด้วย %)
TEACHER_MODEL_ID = "Qwen/Qwen3-14B" #@param ["Qwen/Qwen3-8B", "Qwen/Qwen3-14B", "Qwen/Qwen3-32B", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-14B", "Qwen/Qwen2.5-32B"]
REDUCTION_PERCENTAGE = 35 #@param {type:"slider", min:10, max:50, step:5}
OUTPUT_DIR = "/content/zuce-extracted-specialist"

MAX_SAMPLES = 24
MAX_LENGTH = 512
MIN_RETENTION = 0.60
SEED = 42

print(f"🎯 Selected Model: {TEACHER_MODEL_ID}")
print(f"✂️ Reduction Target: -{REDUCTION_PERCENTAGE}% (Retaining {100 - REDUCTION_PERCENTAGE}%)")

In [ ]:
import json
import os
import shutil
from pathlib import Path
import torch
from transformers import AutoConfig, BitsAndBytesConfig

cfg = AutoConfig.from_pretrained(TEACHER_MODEL_ID, trust_remote_code=False)

hidden_size = cfg.hidden_size
intermediate_size = cfg.intermediate_size
num_layers = cfg.num_hidden_layers
vocab_size = cfg.vocab_size

mlp_params = 3 * hidden_size * intermediate_size * num_layers
attn_params = 4 * (hidden_size * hidden_size) * num_layers
embed_params = vocab_size * hidden_size
estimated_total_params = mlp_params + attn_params + embed_params

MAX_PARAMETERS = int(estimated_total_params * (1.0 - REDUCTION_PERCENTAGE / 100.0))

print(json.dumps({
    "model": TEACHER_MODEL_ID,
    "model_type": cfg.model_type,
    "layers": num_layers,
    "hidden_size": hidden_size,
    "original_mlp_width": intermediate_size,
    "estimated_original_params": f"{estimated_total_params / 1e9:.2f} B",
    "target_reduction": f"-{REDUCTION_PERCENTAGE}%",
    "computed_target_budget": f"{MAX_PARAMETERS / 1e9:.2f} B ({MAX_PARAMETERS:,} params)",
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))

supported_dense_types = {"qwen2", "qwen3", "llama", "mistral", "gemma"}
if cfg.model_type not in supported_dense_types:
    raise RuntimeError(f"{cfg.model_type=} is not supported for physical extraction in ZUCE v0.1")
if "moe" in cfg.model_type.lower():
    raise RuntimeError("MoE models are not supported for physical extraction in ZUCE v0.1")

In [ ]:
CODING_TARGET = [
    {"messages": [{"role": "user", "content": "Write a clean Python function that validates an email address and explain the edge cases."}]},
    {"messages": [{"role": "user", "content": "Implement binary search in Python with type hints and tests."}]},
    {"messages": [{"role": "user", "content": "Refactor this loop into readable Python and explain the complexity."}]},
    {"messages": [{"role": "user", "content": "Write a FastAPI endpoint that accepts JSON, validates input, and returns errors safely."}]},
    {"messages": [{"role": "user", "content": "Debug a Python function that mutates a default list argument."}]},
    {"messages": [{"role": "user", "content": "Create a pytest test suite for a small calculator module."}]},
]

CONTRASTS = {
    "math": [
        "Solve the equation 3x + 7 = 31 step by step.",
        "Find the derivative of x^3 + 2x^2 - 5.",
        "Compute the area of a circle with radius 12.",
    ],
    "translation": [
        "Translate this sentence into Thai: The weather is beautiful today.",
        "Translate this paragraph into English: ฉันกำลังเรียนรู้การเขียนโปรแกรม",
    ],
    "general": [
        "Summarize the causes of seasonal weather changes.",
        "Explain why regular exercise is useful for health.",
    ],
}

len(CODING_TARGET), {name: len(value) for name, value in CONTRASTS.items()}

In [ ]:
from zuce import CapabilitySpec, ParameterBudget, ZUCE

output_path = Path(OUTPUT_DIR)
if output_path.exists():
    raise FileExistsError(f"Output already exists: {output_path}. Change OUTPUT_DIR or remove it manually.")

print(f"🚀 Starting ZUCE Extraction with target budget: {MAX_PARAMETERS / 1e9:.2f} B (-{REDUCTION_PERCENTAGE}%)...")
result = ZUCE.extract(
    model=TEACHER_MODEL_ID,
    capability=CapabilitySpec(
        name="coding",
        target=CODING_TARGET,
        contrasts=CONTRASTS,
    ),
    budget=ParameterBudget(max_parameters=MAX_PARAMETERS),
    output_dir=OUTPUT_DIR,
    device="auto",
    dtype="bfloat16",
    trust_remote_code=False,
    max_samples=MAX_SAMPLES,
    max_length=MAX_LENGTH,
    min_retention=MIN_RETENTION,
    seed=SEED,
)

print(json.dumps(result.to_dict(), indent=2))

In [ ]:
proof = ZUCE.verify(OUTPUT_DIR)
print(json.dumps(proof, indent=2))

for filename in ["zuce_manifest.json", "zero_update_proof.json", "evaluation_report.json"]:
    path = Path(OUTPUT_DIR) / filename
    print("\n==", filename, "==")
    print(path.read_text(encoding="utf-8")[:4000])

## ⚡ โหลดโมเดลแบบ Quantization 8-bit (INT8) และ 4-bit (NF4)
ลด VRAM ลงอีก **50% - 75%** เพื่อให้สามารถรันบน GPU ฟรี Colab T4 (15GB) หรือ RTX 3060/4060 ได้อย่างสบาย

In [ ]:
#@title 🚀 เลือกระดับ Quantization (8-bit vs 4-bit NF4)
QUANT_PRECISION = "4-bit (NF4 with Double Quant)" #@param ["8-bit (INT8)", "4-bit (NF4 with Double Quant)", "Unquantized (BF16/FP16)"]

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if QUANT_PRECISION == "8-bit (INT8)":
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True
    )
    print("⚡ Loading Specialist in 8-bit (INT8) Precision...")
elif QUANT_PRECISION == "4-bit (NF4 with Double Quant)":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    )
    print("⚡ Loading Specialist in 4-bit (NF4 with Double Quantization)...")
else:
    bnb_config = None
    print("⚡ Loading Specialist in Native Precision (BF16/FP16)...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if bnb_config is None else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=False,
)
model.eval()

if torch.cuda.is_available():
    vram_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"✅ Model Loaded successfully! Peak VRAM: {vram_mb:.2f} MB ({vram_mb/1024:.2f} GB)")

In [ ]:
# Quick smoke test: load the extracted HF model and ask for code.
messages = [{
    "role": "user",
    "content": (
        "Return only Python code. Define `chunked(iterable, size)` as a generator "
        "that yields lists of length `size`, with a final shorter list if needed. "
        "Raise ValueError when size <= 0. Include two short pytest tests."
    ),
}]

template_kwargs = dict(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
try:
    inputs = tokenizer.apply_chat_template(**template_kwargs, enable_thinking=False).to(model.device)
except TypeError:
    inputs = tokenizer.apply_chat_template(**template_kwargs).to(model.device)

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
if "</think>" in answer:
    answer = answer.split("</think>", 1)[1].strip()
print(answer)

In [ ]:
# Package the artifact for download or Drive upload.
archive = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Created", archive)

# Optional Google Drive copy:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2(archive, '/content/drive/MyDrive/')